In [44]:
import napari
import os
import numpy as np
import copy
from scipy.spatial import distance, KDTree
from napari import layers

In [2]:
viewer = napari.Viewer()

In [3]:
root = r'E:\Data\BiAPoL\Microwells\20250512\20250512_well1_BF_3Zplanes_full_1'
n_images = 20

chosen_indeces = np.unique(np.random.randint(0, len(os.listdir(root)), n_images))
files_to_chose = np.array(os.listdir(root))

viewer.layers.clear()
for idx, file in enumerate(files_to_chose):
    viewer.open(os.path.join(root, file), plugin='napari-autoscape', downsample=8)

In [4]:
from scipy.spatial import distance, KDTree

In [5]:
fiducial_layer = viewer.layers['Points']
fiducials = fiducial_layer.data * fiducial_layer.scale
fiducial_indeces = [0, 1, 2, 3]

# Distance between fiducials and set radius
dist_matrix = distance.cdist(fiducials, fiducials, 'euclidean')

max_dist = np.max(dist_matrix[dist_matrix > 0])
radius = 9/10 * max_dist
tree = KDTree(fiducials)

# find point with 4 neighbors
fiducial_ids = {}

for i in range(len(fiducials)):
    neighbors = tree.query_ball_point(fiducials[i], radius)
    if len(neighbors) >= 4:
        fiducial_ids['center'] = i
        break

# find south fiducial to center
distances = dist_matrix[fiducial_ids['center']]
fiducial_ids['south'] = np.argwhere(distances == sorted(distances)[1])[0][0]
fiducial_indeces = [index for index in fiducial_indeces if index not in fiducial_ids.values()]

# find fiductial on the west-hand side of the vector from south to center
south_fiducial_coords = fiducials[fiducial_ids['south']]
center_fiducial_coords = fiducials[fiducial_ids['center']]

vector1 = (south_fiducial_coords - center_fiducial_coords)
vector2 = (fiducials[fiducial_indeces[0]] - center_fiducial_coords)
angle = np.arccos(np.dot(vector1, vector2) / (np.linalg.norm(vector1) * np.linalg.norm(vector2))) / np.pi * 180

if angle < 180:
    fiducial_ids['west'] = fiducial_indeces[0]
    fiducial_ids['east'] = fiducial_indeces[1]
else:
    fiducial_ids['west'] = fiducial_indeces[1]
    fiducial_ids['east'] = fiducial_indeces[0]

fiducials = {key: fiducials[fiducial_ids[key]] for key in fiducial_ids.keys()}

# there are four possible orientations of the fiducials: 0, 90, 180, 270 degrees
def get_fiducial_orientation(fiducials):
    if fiducials['west'][1] < fiducials['center'][1] and fiducials['east'][1] > fiducials['center'][1]:
        return 90
    elif fiducials['center'][1] < fiducials['west'][1] and fiducials['center'][1] < fiducials['east'][1]:
        return 180
    elif fiducials['center'][1] > fiducials['west'][1] and fiducials['center'][1] > fiducials['east'][1]:
        return 0
    elif fiducials['west'][1] > fiducials['center'][1] and fiducials['east'][1] < fiducials['center'][1]:
        return 90

orientation = get_fiducial_orientation(fiducials)
orientation

90

In [11]:
fiducials

{'center': array([ 1923.57      ,  7691.37656712, 34139.80192052]),
 'south': array([ 1923.57      ,  7739.78917895, 31600.2969177 ]),
 'west': array([ 1923.57      ,  4346.95043705, 37275.91191483]),
 'east': array([ 1923.57      , 10955.06703728, 37366.6675279 ])}

In [8]:
center_fiducial_coords

array([ 1923.57      ,  7691.37656712, 34139.80192052])

In [10]:
fiducial_ids['south']

1

In [12]:
_vector1 = np.stack([fiducials['center'], fiducials['south'] - fiducials['center']])
_vector2  = np.stack([fiducials['center'], fiducials['west'] - fiducials['center']])
_vector3  = np.stack([fiducials['center'], fiducials['east'] - fiducials['center']])
viewer.add_vectors(_vector1, name='vector1', edge_color='red', edge_width=50, vector_style='arrow', blending='translucent_no_depth')
viewer.add_vectors(_vector2, name='vector2', edge_color='green', edge_width=50, vector_style='arrow', blending='translucent_no_depth')
viewer.add_vectors(_vector3, name='vector3', edge_color='blue', edge_width=50, vector_style='arrow', blending='translucent_no_depth')

<Vectors layer 'vector3' at 0x1b513d94f50>

In [13]:
# Length and width of microwell plate
columns = 28      #number of microwell rows
rows = 32       #number of microwell colummns
microwell_size_um = 290    #size of single microwell in micrometers

In [ ]:
row_vector = (fiducials['east'] - fiducials['west']) / (columns - 1)

def rotate_vector(vector, angle):
    """Rotate a vector by a given angle in degree."""
    angle = np.deg2rad(angle)  # Convert angle to radians
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle), np.cos(angle)]])
    return R @ vector

col_vector1 = rotate_vector(row_vector[1:], -120)
col_vector1 = np.insert(col_vector1, 0, 0)
col_vector2 = rotate_vector(row_vector[1:], -60)
col_vector2 = np.insert(col_vector2, 0, 0)

row_vector1 = np.stack([fiducials['west'], row_vector])
col_vector1 = np.stack([fiducials['west'], col_vector1])
col_vector2 = np.stack([fiducials['west'], col_vector2])

In [40]:
well_positions = []
for j in range(rows):

    if j % 4 == 1 or j % 4 == 3:
        delta = col_vector1
    else:
        delta = np.zeros_like(col_vector1)

    starting_position = fiducials['west'] + j//2 * col_vector1[1:] + j//2 * col_vector2[1:] + delta[1:]
    for i in range(columns):
        well_positions.append(starting_position + i * row_vector1[1:])

well_positions = np.stack(well_positions).squeeze()

In [43]:
def hexagon_around_point(point, size):
    """Generate coordinates of a hexagon around a point."""
    # consider that points are zyx and disregard z
    z_value = point[0]  # Save z coordinate 
    point = point[1:]  # Take only x and y coordinates
    angles = np.linspace(0, 2 * np.pi, 7)[:-1] + np.pi / 6  # 6 angles for hexagon, offset by 30 degrees
    points =  point + size * np.column_stack((np.cos(angles), np.sin(angles)))

    # put z back in
    points = np.insert(points, 0, z_value, axis=1)  # Insert z coordinate back
    return points

hexagons = [hexagon_around_point(p, size=100) for p in well_positions]

viewer.add_shapes(hexagons, edge_color='red', face_color='transparent', name='hexagon_west', shape_type='polygon', edge_width=2)

<Shapes layer 'hexagon_west' at 0x1b6d4a4bed0>

In [39]:
well_positions[0]

array([[ 1923.57      ,  4346.95043705, 37275.91191483]])

In [23]:
viewer.add_vectors(row_vector1, name='row_vector', edge_color='red', edge_width=50, vector_style='arrow', blending='translucent_no_depth')
viewer.add_vectors(col_vector1, name='col_vector', edge_color='green', edge_width=50, vector_style='arrow', blending='translucent_no_depth')
viewer.add_vectors(col_vector2, name='col_vector2', edge_color='blue', edge_width=50, vector_style='arrow', blending='translucent_no_depth')
viewer.add_points(np.stack(well_positions).squeeze(), size=100, face_color='yellow', name='well_positions')

<Points layer 'well_positions' at 0x1b5042e34d0>

In [ ]:
def create_well_grid(fiducial_layer: layers.Points, ncolumns: int = 28, nrows: int = 32, microwell_diameter_um: float = 200):
    """
    Create a grid of wells based on fiducial positions.
    
    Parameters:
        fiducial_positions: Coordinates of fiducials.
        ncolumns: Number of columns in the microwell plate.
        nrows: Number of rows in the microwell plate.
        microwell_size_um: Size of a single microwell in micrometers.
        
    Returns:
        Well positions as a numpy array.
    """
    # Calculate row and column vectors
    fiducial_layer = viewer.layers['Points']
    fiducials = fiducial_layer.data * fiducial_layer.scale
    fiducial_indeces = [0, 1, 2, 3]

    # Distance between fiducials and set radius
    dist_matrix = distance.cdist(fiducials, fiducials, 'euclidean')

    max_dist = np.max(dist_matrix[dist_matrix > 0])
    radius = 9/10 * max_dist
    tree = KDTree(fiducials)

    # find point with 4 neighbors
    fiducial_ids = {}

    for i in range(len(fiducials)):
        neighbors = tree.query_ball_point(fiducials[i], radius)
        if len(neighbors) >= 4:
            fiducial_ids['center'] = i
            break

    # find south fiducial to center
    distances = dist_matrix[fiducial_ids['center']]
    fiducial_ids['south'] = np.argwhere(distances == sorted(distances)[1])[0][0]
    fiducial_indeces = [index for index in fiducial_indeces if index not in fiducial_ids.values()]

    # find fiductial on the west-hand side of the vector from south to center
    south_fiducial_coords = fiducials[fiducial_ids['south']]
    center_fiducial_coords = fiducials[fiducial_ids['center']]

    vector1 = (south_fiducial_coords - center_fiducial_coords)
    vector2 = (fiducials[fiducial_indeces[0]] - center_fiducial_coords)
    angle = np.arccos(np.dot(vector1, vector2) / (np.linalg.norm(vector1) * np.linalg.norm(vector2))) / np.pi * 180

    if angle < 180:
        fiducial_ids['west'] = fiducial_indeces[0]
        fiducial_ids['east'] = fiducial_indeces[1]
    else:
        fiducial_ids['west'] = fiducial_indeces[1]
        fiducial_ids['east'] = fiducial_indeces[0]

    fiducials = {key: fiducials[fiducial_ids[key]] for key in fiducial_ids.keys()}

    # there are four possible orientations of the fiducials: 0, 90, 180, 270 degrees
    def get_fiducial_orientation(fiducials):
        if fiducials['west'][1] < fiducials['center'][1] and fiducials['east'][1] > fiducials['center'][1]:
            return 90
        elif fiducials['center'][1] < fiducials['west'][1] and fiducials['center'][1] < fiducials['east'][1]:
            return 180
        elif fiducials['center'][1] > fiducials['west'][1] and fiducials['center'][1] > fiducials['east'][1]:
            return 0
        elif fiducials['west'][1] > fiducials['center'][1] and fiducials['east'][1] < fiducials['center'][1]:
            return 90

    orientation = get_fiducial_orientation(fiducials)
    orientation
    
    row_vector = (fiducials['east'] - fiducials['west']) / (columns - 1)

    def rotate_vector(vector, angle):
        """Rotate a vector by a given angle in degree."""
        angle = np.deg2rad(angle)  # Convert angle to radians
        R = np.array([[np.cos(angle), -np.sin(angle)],
                    [np.sin(angle), np.cos(angle)]])
        return R @ vector

    col_vector1 = rotate_vector(row_vector[1:], -120)
    col_vector1 = np.insert(col_vector1, 0, 0)
    col_vector2 = rotate_vector(row_vector[1:], -60)
    col_vector2 = np.insert(col_vector2, 0, 0)

    row_vector1 = np.stack([fiducials['west'], row_vector])
    col_vector1 = np.stack([fiducials['west'], col_vector1])
    col_vector2 = np.stack([fiducials['west'], col_vector2])

    well_positions = []
    for j in range(rows):

        if j % 4 == 1 or j % 4 == 3:
            delta = col_vector1
        else:
            delta = np.zeros_like(col_vector1)

        starting_position = fiducials['west'] + j//2 * col_vector1[1:] + j//2 * col_vector2[1:] + delta[1:]
        for i in range(columns):
            well_positions.append(starting_position + i * row_vector1[1:])

    well_positions = np.stack(well_positions).squeeze()


    hexagons = [hexagon_around_point(p, size=100) for p in well_positions]

    return layers.Shapes(hexagons, edge_color='red', face_color='transparent', name='hexagon_west', shape_type='polygon', edge_width=2)

In [ ]:
    def hexagon_around_point(point, size):
        """Generate coordinates of a hexagon around a point."""
        # consider that points are zyx and disregard z
        z_value = point[0]  # Save z coordinate 
        point = point[1:]  # Take only x and y coordinates
        angles = np.linspace(0, 2 * np.pi, 7)[:-1] + np.pi / 6  # 6 angles for hexagon, offset by 30 degrees
        points =  point + size * np.column_stack((np.cos(angles), np.sin(angles)))

        # put z back in
        points = np.insert(points, 0, z_value, axis=1)  # Insert z coordinate back
        return points
